<a href="https://colab.research.google.com/github/PRUTU29/ML_LAB/blob/main/HPC_JACKFRUIT/jupyter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%cuda
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <math.h>
#include <cuda_runtime.h>

#define N 1024 // Matrix size N x N
#define BLOCK_SIZE 16

// CPU Matrix Multiplication
void matrixMulCPU(float *a, float *b, float *c, int n) {
    for (int i = 0; i < n; ++i) {
        for (int j = 0; j < n; ++j) {
            float sum = 0.0f;
            for (int k = 0; k < n; ++k) {
                sum += a[i * n + k] * b[k * n + j];
            }
            c[i * n + j] = sum;
        }
    }
}

// GPU Matrix Multiplication Kernel (SIMD / Multi-Threading)
__global__ void matrixMulGPU(float *a, float *b, float *c, int n) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < n && col < n) {
        float sum = 0.0f;
        for (int k = 0; k < n; ++k) {
            sum += a[row * n + k] * b[k * n + col];
        }
        c[row * n + col] = sum;
    }
}

// Helper function to print a portion of a matrix
void printMatrix(float *matrix, int n, int limit, const char *name) {
    printf("\n--- %s Matrix (first %d x %d elements) ---\n", name, limit, limit);
    for (int i = 0; i < limit; ++i) {
        for (int j = 0; j < limit; ++j) {
            printf("%.4f ", matrix[i * n + j]);
        }
        printf("\n");
    }
    printf("-------------------------------------------\n");
}

int main() {
    int size = N * N * sizeof(float);
    float *h_a, *h_b, *h_c_cpu, *h_c_gpu;
    float *d_a, *d_b, *d_c;

    // Allocate Host Memory
    h_a = (float *)malloc(size);
    h_b = (float *)malloc(size);
    h_c_cpu = (float *)malloc(size);
    h_c_gpu = (float *)malloc(size);

    // Initialize Matrices
    srand(42);
    for (int i = 0; i < N * N; i++) {
        h_a[i] = (float)rand() / RAND_MAX;
        h_b[i] = (float)rand() / RAND_MAX;
    }

    // Print a portion of the input matrices
    int print_limit = 8; // For example, print top-left 8x8
    if (N < print_limit) { // If N is smaller than print_limit, print the whole matrix
        print_limit = N;
    }
    printMatrix(h_a, N, print_limit, "Matrix A (Input)");
    printMatrix(h_b, N, print_limit, "Matrix B (Input)");

    // Allocate Device Memory
    cudaMalloc((void**)&d_a, size);
    cudaMalloc((void**)&d_b, size);
    cudaMalloc((void**)&d_c, size);

    // Copy data from Host to Device
    cudaMemcpy(d_a, h_a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, size, cudaMemcpyHostToDevice);

    // --- CPU Execution ---
    printf("==========================================\n");
    printf("Matrix Multiplication (Size: %d x %d)\n", N, N);
    printf("==========================================\n");
    printf("\nStarting CPU Execution...\n");
    clock_t start_cpu = clock();
    matrixMulCPU(h_a, h_b, h_c_cpu, N);
    clock_t end_cpu = clock();
    double time_cpu = ((double)(end_cpu - start_cpu)) / CLOCKS_PER_SEC;
    printf("-> CPU Time: %f seconds\n", time_cpu);

    // --- GPU Execution ---
    printf("\nStarting GPU Execution (SIMD / Multi-Threading)...\n");
    dim3 threadsPerBlock(BLOCK_SIZE, BLOCK_SIZE);
    dim3 numBlocks((N + BLOCK_SIZE - 1) / BLOCK_SIZE, (N + BLOCK_SIZE - 1) / BLOCK_SIZE);

    cudaEvent_t start_gpu, stop_gpu;
    cudaEventCreate(&start_gpu);
    cudaEventCreate(&stop_gpu);

    cudaEventRecord(start_gpu);
    matrixMulGPU<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c, N);
    cudaEventRecord(stop_gpu);

    // Synchronize to wait for GPU to finish
    cudaEventSynchronize(stop_gpu);
    float time_gpu_ms = 0;
    cudaEventElapsedTime(&time_gpu_ms, start_gpu, stop_gpu);
    double time_gpu = time_gpu_ms / 1000.0;
    printf("-> GPU Time: %f seconds\n", time_gpu);

    // Copy result back to Host
    cudaMemcpy(h_c_gpu, d_c, size, cudaMemcpyDeviceToHost);

    // Print a portion of the matrices
    // N is too large to print the full matrix, so we print a small corner.
    // print_limit is already defined above.
    printMatrix(h_c_cpu, N, print_limit, "CPU Result");
    printMatrix(h_c_gpu, N, print_limit, "GPU Result");

    // Verify Results
    float max_error = 0.0f;
    for (int i = 0; i < N * N; i++) {
        float diff = fabs(h_c_cpu[i] - h_c_gpu[i]);
        if (diff > max_error) {
            max_error = diff;
        }
    }
    printf("\nMax error between CPU and GPU results: %f\n", max_error);

    // Calculate Speedup
    printf("\n==========================================\n");
    printf("Speedup (CPU_Time / GPU_Time): %.2fX\n", time_cpu / time_gpu);
    printf("==========================================\n");

    // Free Memory
    free(h_a);
    free(h_b);
    free(h_c_cpu);
    free(h_c_gpu);
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);
    cudaEventDestroy(start_gpu);
    cudaEventDestroy(stop_gpu);

    return 0;
}

UsageError: Cell magic `%%cuda` not found.
